# Held-out GlycA main-effect analysis


In [ ]:
import os, json, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd

os.environ['CHD_RUN_ID'] = os.environ.get('CHD_PRIMARY_RUN_ID', 'final_rerun_2026')
os.environ['CHD_COHORT_ID']='prs_raw106_complete'
os.environ['CHD_INNER_THREADS']='1'

shared = Path(os.environ.get("CHD_SHARED_NOTEBOOK", str(Path.cwd() / "00_shared_config.ipynb")))
if not shared.is_file():
    raise FileNotFoundError(shared)
get_ipython().run_line_magic('run',f'"{shared}"')
if CHD_COHORT_ID!='prs_raw106_complete' or ANALYSIS_ROLE!='primary':
    raise RuntimeError('The primary cohort is required.')
print('Loaded:',shared,'| run=',RUN_ID,'| cohort=',CHD_COHORT_ID,flush=True)


In [ ]:
source_filename='result6_metabolite_main_effects_discovery_replication.csv'
main_results_dir=Path(TABLE_DIR).resolve().parent
preferred=main_results_dir/'result6_exploratory_metabolites'/'tables'/source_filename
legacy=Path(TABLE_DIR).resolve()/source_filename
if preferred.is_file():
    source=preferred
elif legacy.is_file():
    source=legacy
else:
    matches=sorted({p.resolve() for p in main_results_dir.rglob(source_filename) if p.is_file()})
    if len(matches)!=1:
        raise FileNotFoundError(f'Expected one {source_filename} in the primary run, found {len(matches)}: {matches}')
    source=matches[0]
source=Path(source).resolve()
print('Locked Result 6 main-effect source:',source,flush=True)

d=pd.read_csv(source)
required={'Metabolite','Metabolite_Label','coef','p','FDR','Replication_Beta','Replication_P','Replication_FDR','Fit_Error','Replication_Fit_Error'}
missing=required.difference(d.columns)
if missing:
    raise RuntimeError('Missing main-effect columns: '+','.join(sorted(missing)))
if len(d)!=106 or d.Metabolite.astype(str).nunique()!=106:
    raise RuntimeError('The Result 6 main-effect table must contain 106 unique metabolites.')
for c in ['coef','p','FDR','Replication_Beta','Replication_P','Replication_FDR']:
    d[c]=pd.to_numeric(d[c],errors='coerce')
fit_ok=(d.Fit_Error.fillna('').astype(str).str.strip().eq('') & d.Replication_Fit_Error.fillna('').astype(str).str.strip().eq(''))
same_direction=np.sign(d.coef).eq(np.sign(d.Replication_Beta))
candidate=d.loc[fit_ok & same_direction & d.FDR.lt(.05) & d.Replication_FDR.lt(.05)].copy()
candidate['Metabolite']=candidate.Metabolite.astype(str)
if len(candidate)!=1:
    raise RuntimeError(f'Expected one GlycA candidate, found {len(candidate)}: {candidate[["Metabolite","Metabolite_Label"]].to_dict("records")}')
if candidate.Metabolite_Label.iloc[0].strip().lower()!='glyca':
    raise RuntimeError('GlycA must be the sole candidate before opening the test split.')
field=str(candidate.Metabolite.iloc[0])
if field not in set(map(str,nmr106_cols)):
    raise RuntimeError('GlycA is absent from the locked Raw106 panel.')

OUT=Path(RESULTS_OUTPUT_DIR)/'result6_glyca_test_confirmation_v1'
OUT_TABLE=OUT/'tables'
OUT_TABLE.mkdir(parents=True,exist_ok=True)
lock_table=candidate[['Metabolite','Metabolite_Label','coef','p','FDR','Replication_Beta','Replication_P','Replication_FDR']].rename(columns={'coef':'Train_Beta','p':'Train_P','FDR':'Train_FDR_BH_106','Replication_Beta':'Validation_Beta','Replication_P':'Validation_P','Replication_FDR':'Validation_FDR_BH_106'})
lock_table.to_csv(OUT_TABLE/'candidate_lock_glyca_before_test.csv',index=False)
lock_payload={
    'specification_id':'result6_glyca_main_effect_test_confirmation_v1',
    'pipeline_manifest_sha256':current_pipeline_sha256(),
    'source_sha256':sha256_file(source),
    'candidate_rule':'Train BH-FDR<0.05 AND validation BH-FDR<0.05 AND same direction; exactly one candidate GlycA',
    'candidate_field':field,
    'model':'Clinical_plus_PRS + GlycA main effect',
    'preprocessing':'training-only imputation and StandardScaler',
    'estimator':'unpenalized Cox PH with robust SE; Efron ties; batch mode',
    'confirmation_rule':'two-sided test P<0.05 and same direction in train, validation and test',
    'multiple_testing':'one prelocked candidate; BH-FDR equals raw P',
    'external_replication':False,
}
lock_sig=canonical_json_digest(lock_payload)
lock_path=OUT/'CANDIDATE_LOCK_GLYCA_BEFORE_TEST.json'
if lock_path.exists():
    old=json.loads(lock_path.read_text(encoding='utf-8'))
    if old.get('signature')!=lock_sig or old.get('signature_payload')!=lock_payload:
        raise RuntimeError('The existing GlycA lock differs from the current specification.')
    print('RESUME identical GlycA candidate lock:',lock_path,flush=True)
else:
    atomic_write_json({'locked_at':str(pd.Timestamp.now()),'signature':lock_sig,'signature_payload':lock_payload},lock_path)
    print('LOCKED GLYCA BEFORE TEST FITTING:',lock_path,flush=True)
display(lock_table)


In [ ]:
df_train,df_val,df_test,final_feature_sets,pipeline_manifest=validate_pipeline_artifacts(load_frames=True)
if current_pipeline_sha256()!=lock_payload['pipeline_manifest_sha256']:
    raise RuntimeError('The pipeline manifest changed after candidate selection.')
base_features=list(final_feature_sets['Clinical_plus_PRS'])
if PRS_COL not in base_features or field in base_features:
    raise RuntimeError('Unexpected Clinical_plus_PRS covariates.')
features=list(dict.fromkeys(base_features+[field]))
X_train,X_val,X_test,actual_columns=prepare_X_train_val_test(df_train,df_val,df_test,features,scale=True)
actual_columns=list(actual_columns)
if field not in actual_columns or PRS_COL not in actual_columns:
    raise RuntimeError('PRS or GlycA is missing after preprocessing.')
if not np.isfinite(X_test.to_numpy(float)).all():
    raise RuntimeError('The GlycA test design contains non-finite values.')
fit_frame=pd.concat([df_test[['Time','Event']].reset_index(drop=True),X_test.reset_index(drop=True)],axis=1)
design_payload={
    'lock_signature':lock_sig,
    'pipeline_manifest_sha256':current_pipeline_sha256(),
    'test_eid_sha256':eid_digest(df_test),
    'test_outcome_sha256':dataframe_digest(df_test,['Eid','Time','Event']),
    'columns':actual_columns,
    'test_n':int(len(df_test)),
    'test_event_n':int(df_test.Event.sum()),
}
design_sig=canonical_json_digest(design_payload)
print(f'Test design ready: N={len(df_test):,}; events={int(df_test.Event.sum()):,}; features={len(actual_columns)}',flush=True)


In [ ]:
from lifelines import CoxPHFitter
from lifelines.exceptions import ConvergenceWarning
from lifelines.statistics import proportional_hazard_test

result_path=OUT_TABLE/'result6_glyca_main_effect_test_confirmation.csv'
cache_path=OUT/'GLYCA_TEST_MODEL.json'
fit_signature_payload={
    'specification_id':'result6_glyca_main_effect_test_confirmation_v1',
    'lock_signature':lock_sig,
    'design_signature':design_sig,
    'penalizer':0.0,
    'robust':True,
    'ties':'efron',
    'batch_mode':True,
    'ph_time_transform':'rank',
}
fit_sig=canonical_json_digest(fit_signature_payload)
resume=False
if result_path.exists() and cache_path.exists():
    saved=json.loads(cache_path.read_text(encoding='utf-8'))
    if saved.get('signature')==fit_sig and saved.get('signature_payload')==fit_signature_payload:
        result=pd.read_csv(result_path)
        if len(result)==1 and result.Metabolite.astype(str).iloc[0]==field:
            resume=True
            print('RESUME valid signed GlycA test result:',result_path,flush=True)
if not resume:
    started=time.time()
    print(f'START GlycA test Cox | N={len(fit_frame):,} | events={int(fit_frame.Event.sum()):,} | features={len(actual_columns)}',flush=True)
    model=CoxPHFitter(penalizer=0.0,l1_ratio=0.0)
    with warnings.catch_warnings():
        warnings.simplefilter('error',ConvergenceWarning)
        model.fit(fit_frame,duration_col='Time',event_col='Event',robust=True,batch_mode=True,show_progress=False)
    z=model.summary.loc[field]
    print(f'COX FIT DONE | elapsed={(time.time()-started)/60:.1f} min | starting PH diagnostic',flush=True)
    ph=proportional_hazard_test(model,fit_frame,time_transform='rank').summary
    ph_p=float(ph.loc[field,'p'])
    ph_test_n=int(ph['p'].notna().sum())
    ph_p_bonf=min(1.0,ph_p*ph_test_n)
    train_beta=float(lock_table.Train_Beta.iloc[0])
    validation_beta=float(lock_table.Validation_Beta.iloc[0])
    test_beta=float(z['coef'])
    same_direction=bool(np.sign(train_beta)==np.sign(validation_beta)==np.sign(test_beta))
    test_p=float(z['p'])
    confirmed=bool(test_p<.05 and same_direction)
    row={
        'Metabolite':field,
        'Metabolite_Label':'GlycA',
        'Train_Beta':train_beta,
        'Train_P':float(lock_table.Train_P.iloc[0]),
        'Train_FDR_BH_106':float(lock_table.Train_FDR_BH_106.iloc[0]),
        'Validation_Beta':validation_beta,
        'Validation_P':float(lock_table.Validation_P.iloc[0]),
        'Validation_FDR_BH_106':float(lock_table.Validation_FDR_BH_106.iloc[0]),
        'Test_N':int(len(df_test)),
        'Test_Event_N':int(df_test.Event.sum()),
        'Test_Beta':test_beta,
        'Test_SE':float(z['se(coef)']),
        'Test_HR':float(z['exp(coef)']),
        'Test_HR_CI_Lower':float(z['exp(coef) lower 95%']),
        'Test_HR_CI_Upper':float(z['exp(coef) upper 95%']),
        'Test_P':test_p,
        'Test_FDR_BH_1':test_p,
        'Three_Split_Same_Direction':same_direction,
        'Heldout_Internal_Confirmation':confirmed,
        'PH_P_Raw_GlycA':ph_p,
        'PH_Tested_Term_N':ph_test_n,
        'PH_P_Bonferroni_All_Terms':ph_p_bonf,
        'Elapsed_Minutes_Including_PH':float((time.time()-started)/60),
    }
    result=pd.DataFrame([row])
    result.to_csv(result_path,index=False)
    atomic_write_json({'signature':fit_sig,'signature_payload':fit_signature_payload,'row':row},cache_path)
    print(f'PH DIAGNOSTIC DONE | total elapsed={(time.time()-started)/60:.1f} min',flush=True)
display(result)


In [ ]:
r=result.iloc[0]
status={
    'status':'SUCCESS',
    'analysis':'GlycA main-effect held-out internal confirmation',
    'candidate_n':1,
    'test_n':int(r.Test_N),
    'test_event_n':int(r.Test_Event_N),
    'test_p':float(r.Test_P),
    'confirmed':bool(r.Heldout_Internal_Confirmation),
    'three_split_same_direction':bool(r.Three_Split_Same_Direction),
    'ph_p_bonferroni_all_terms':float(r.PH_P_Bonferroni_All_Terms),
    'candidate_lock_signature':lock_sig,
    'result_table':str(result_path),
    'terminology':'held-out internally confirmed; not external replication',
}
atomic_write_json(status,OUT/'SUCCESS.json')
display(result)
